# Moment-tensor conventions, checked against first principles

Every moment tensor in seismo-sbi is the vector `m6 = [m_rr, m_tt, m_pp, m_rt, m_rp, m_tp]` in N m, in the spherical basis $r, \theta, \phi$ = up, south, east. This is the convention of the Global CMT catalogue and of Instaseis. This notebook checks that convention numerically, at each place it matters:

- the tensor of a fault given by strike, dip and rake;
- the far-field P radiation pattern and first motions;
- the radiation geometry of the Instaseis forward model;
- the position of standard sources on the lune;
- the scalar moment;
- how horizontal components are stored and oriented.

In [1]:
import os

import numpy as np
from obspy import Stream, Trace, UTCDateTime
from obspy.core.inventory import Channel, Inventory, Network, Station
from pyrocko import moment_tensor as pmt
from pyrocko import orthodrome

from seismo_sbi.data_handling.preprocessing.processing import rotate_horizontals_to_north_east
from seismo_sbi.moment_tensor.comparison import from_pyrocko, pyrocko_mt
from seismo_sbi.moment_tensor.conventions import scalar_moment
from seismo_sbi.moment_tensor.lune_angles import mts6_to_gamma_delta
from seismo_sbi.simulators.instaseis.simulator import InstaseisSourceSimulator
from seismo_sbi.simulators.receivers import Receiver, Receivers
from seismo_sbi.simulators.simulation_io import component_alias

INSTASEIS_DB = os.environ["INSTASEIS_DB"]
SCALAR_MOMENT_NM = 1e17

## A fault as a moment tensor

Aki & Richards (2002, box 4.4) give the tensor of a shear dislocation with strike $\phi_s$, dip $\delta$ and rake $\lambda$ in north-east-down coordinates $x, y, z$. The up-south-east components follow from $r = -z$, $\theta = -x$, $\phi = y$: $m_{rr} = M_{zz}$, $m_{tt} = M_{xx}$, $m_{pp} = M_{yy}$, $m_{rt} = M_{xz}$, $m_{rp} = -M_{yz}$, $m_{tp} = -M_{xy}$. The textbook formula, pyrocko's construction and the strike, dip and rake read back through `pyrocko_mt` all agree.

In [2]:
def aki_richards_m6(strike_deg, dip_deg, rake_deg, scalar_moment_nm):
    '''``m6`` (up-south-east, N m) of a shear fault, from Aki & Richards' north-east-down tensor.'''
    strike, dip, rake = np.radians([strike_deg, dip_deg, rake_deg])
    m_xx = -(np.sin(dip) * np.cos(rake) * np.sin(2 * strike) + np.sin(2 * dip) * np.sin(rake) * np.sin(strike) ** 2)
    m_xy = np.sin(dip) * np.cos(rake) * np.cos(2 * strike) + 0.5 * np.sin(2 * dip) * np.sin(rake) * np.sin(2 * strike)
    m_xz = -(np.cos(dip) * np.cos(rake) * np.cos(strike) + np.cos(2 * dip) * np.sin(rake) * np.sin(strike))
    m_yy = np.sin(dip) * np.cos(rake) * np.sin(2 * strike) - np.sin(2 * dip) * np.sin(rake) * np.cos(strike) ** 2
    m_yz = -(np.cos(dip) * np.cos(rake) * np.sin(strike) - np.cos(2 * dip) * np.sin(rake) * np.cos(strike))
    m_zz = np.sin(2 * dip) * np.sin(rake)
    return scalar_moment_nm * np.array([m_zz, m_xx, m_yy, m_xz, -m_yz, -m_xy])


faults = {"left-lateral, N-S vertical plane": (0, 90, 0), "thrust striking east": (90, 45, 90),
          "oblique normal": (30, 60, -80)}
for name, (strike_deg, dip_deg, rake_deg) in faults.items():
    textbook = aki_richards_m6(strike_deg, dip_deg, rake_deg, SCALAR_MOMENT_NM)
    pyrocko = from_pyrocko(pmt.MomentTensor(strike=strike_deg, dip=dip_deg, rake=rake_deg, scalar_moment=SCALAR_MOMENT_NM))
    planes = pyrocko_mt(textbook).both_strike_dip_rake()
    print(f"{name:34s} m6 / M0 = {np.round(textbook / SCALAR_MOMENT_NM, 3) + 0.0}")
    print(f"{'':34s} planes read back: {[tuple(int(round(angle)) for angle in plane) for plane in planes]}")
    assert np.allclose(textbook, pyrocko, atol=1e-9 * SCALAR_MOMENT_NM)
    assert any(np.allclose(np.array(plane) % 360, np.array([strike_deg, dip_deg, rake_deg]) % 360, atol=0.5) for plane in planes)

left-lateral, N-S vertical plane   m6 / M0 = [ 0.  0.  0.  0.  0. -1.]
                                   planes read back: [(90, 90, 180), (0, 90, 0)]
thrust striking east               m6 / M0 = [ 1. -1.  0.  0.  0.  0.]
                                   planes read back: [(90, 45, 90), (270, 45, 90)]
oblique normal                     m6 / M0 = [-0.853  0.083  0.77  -0.321 -0.383  0.294]
                                   planes read back: [(30, 60, -80), (191, 31, -107)]


## Far-field P radiation and first motions

In a whole space, the far-field P displacement along the ray direction $\hat\gamma$ is proportional to $\hat\gamma^\mathsf{T} M \hat\gamma$. For a fault with unit normal $\hat n$ and unit slip $\hat s$, $M = M_0(\hat n \hat s^\mathsf{T} + \hat s \hat n^\mathsf{T})$, so the first motion has the sign of $(\hat n \cdot \hat\gamma)(\hat s \cdot \hat\gamma)$, positive (compressional) or negative (dilatational). Below, rays leave the oblique normal fault at 36 azimuths and three take-off angles. The sign of $\hat\gamma^\mathsf{T} M \hat\gamma$, with $M$ built from `m6` in up-south-east, is compared with the fault-vector prediction (Aki & Richards eq. 4.88 for $\hat n$ and $\hat s$).

In [3]:
def use_matrix(m6):
    '''The symmetric 3x3 tensor of ``m6`` in the up-south-east basis.'''
    m_rr, m_tt, m_pp, m_rt, m_rp, m_tp = m6
    return np.array([[m_rr, m_rt, m_rp], [m_rt, m_tt, m_tp], [m_rp, m_tp, m_pp]])


def ray_direction_use(azimuth_deg, takeoff_deg):
    '''Unit ray direction (up, south, east) leaving the source; take-off measured from straight down.'''
    azimuth, takeoff = np.radians(azimuth_deg), np.radians(takeoff_deg)
    return np.array([-np.cos(takeoff), -np.sin(takeoff) * np.cos(azimuth), np.sin(takeoff) * np.sin(azimuth)])


def normal_and_slip_use(strike_deg, dip_deg, rake_deg):
    '''Fault normal and slip vectors (Aki & Richards 4.88, north-east-down) in up-south-east.'''
    strike, dip, rake = np.radians([strike_deg, dip_deg, rake_deg])
    normal_ned = np.array([-np.sin(dip) * np.sin(strike), np.sin(dip) * np.cos(strike), -np.cos(dip)])
    slip_ned = np.array([np.cos(rake) * np.cos(strike) + np.cos(dip) * np.sin(rake) * np.sin(strike),
                         np.cos(rake) * np.sin(strike) - np.cos(dip) * np.sin(rake) * np.cos(strike),
                         -np.sin(rake) * np.sin(dip)])
    ned_to_use = np.array([[0, 0, -1], [-1, 0, 0], [0, 1, 0]])
    return ned_to_use @ normal_ned, ned_to_use @ slip_ned


strike_deg, dip_deg, rake_deg = faults["oblique normal"]
tensor = use_matrix(aki_richards_m6(strike_deg, dip_deg, rake_deg, SCALAR_MOMENT_NM))
normal, slip = normal_and_slip_use(strike_deg, dip_deg, rake_deg)
agreeing, compressional = 0, 0
rays = [(azimuth_deg, takeoff_deg) for azimuth_deg in range(0, 360, 10) for takeoff_deg in (30, 60, 89)]
for azimuth_deg, takeoff_deg in rays:
    ray = ray_direction_use(azimuth_deg, takeoff_deg)
    radiation = ray @ tensor @ ray
    agreeing += np.sign(radiation) == np.sign((normal @ ray) * (slip @ ray))
    compressional += radiation > 0
print(f"{agreeing} of {len(rays)} first motions agree with the fault-vector prediction; {compressional} are compressional")
assert agreeing == len(rays)

mirrored = tensor * np.array([[1, 1, -1], [1, 1, -1], [-1, -1, 1]])
mirror_agreeing = sum(np.sign(ray_direction_use(*r) @ mirrored @ ray_direction_use(*r))
                      == np.sign((normal @ ray_direction_use(*r)) * (slip @ ray_direction_use(*r))) for r in rays)
print(f"with m_rp and m_tp negated (an east-west mirror), {mirror_agreeing} of {len(rays)} agree")

108 of 108 first motions agree with the fault-vector prediction; 50 are compressional
with m_rp and m_tp negated (an east-west mirror), 82 of 108 agree


## Radiation geometry of the Instaseis forward model

For surface waves, Herrmann's expansion of the vertical displacement gives two ratios that depend only on the station azimuth $\phi$. They do not depend on the Earth model or the frequency band:

$$Z(m_{rp}) / Z(m_{rt}) = -\tan\phi, \qquad Z(m_{tp}) / \big(Z(m_{tt}) - Z(m_{pp})\big) = -\tan 2\phi.$$

Here $Z(m_{ij})$ is the vertical seismogram of a unit $m_{ij}$. An east-west mirror of the forward model reverses both signs. Instaseis is run once per unit component, at four stations 400 km from the source, and each ratio is the least-squares amplitude of one seismogram against the other. For $m_{rr}$ alone the transverse component must vanish, which checks that east and north are not interchanged.

In [4]:
source_latitude_deg, source_longitude_deg = 37.0, -118.0
azimuths_deg = [20, 110, 200, 290]
receivers = []
for azimuth_deg in azimuths_deg:
    latitude, longitude = orthodrome.azidist_to_latlon(source_latitude_deg, source_longitude_deg, azimuth_deg, 400 / 111.19)
    receivers.append(Receiver(float(latitude), float(longitude), "XX", f"A{azimuth_deg:03d}", ["Z", "E", "N"]))
receivers = Receivers(receivers=receivers)
simulator = InstaseisSourceSimulator(
    INSTASEIS_DB, components="ZEN", receivers=receivers, seismogram_duration_in_s=300,
    synthetics_processing={"filter": {"type": "bandpass", "freqmin": 0.01, "freqmax": 0.03, "corners": 4,
                                      "zerophase": False}, "sampling_rate": 1.0, "filter_sampling_rate": 1.0})
unit_seismograms = [simulator.run_simulation({"source_location": [source_latitude_deg, source_longitude_deg, 15.0, 0.0],
                                              "moment_tensor": list(unit)})[1] for unit in np.eye(6) * SCALAR_MOMENT_NM]


def amplitude_ratio(numerator, denominator):
    return float(numerator @ denominator / (denominator @ denominator))


print(f"{'azimuth':>8s}{'Z(m_rp)/Z(m_rt)':>18s}{'-tan(az)':>10s}{'ratio 2':>10s}{'-tan(2az)':>11s}{'T/R, m_rr':>11s}")
for azimuth_deg, receiver in zip(azimuths_deg, receivers.iterate()):
    vertical = [seismograms[receiver.station_name]["Z"] for seismograms in unit_seismograms]
    first = amplitude_ratio(vertical[4], vertical[3])
    second = amplitude_ratio(vertical[5], vertical[1] - vertical[2])
    azimuth = np.radians(azimuth_deg)
    east, north = (unit_seismograms[0][receiver.station_name][component] for component in "EN")
    radial, transverse = north * np.cos(azimuth) + east * np.sin(azimuth), -north * np.sin(azimuth) + east * np.cos(azimuth)
    transverse_ratio = np.std(transverse) / np.std(radial)
    print(f"{azimuth_deg:8d}{first:18.3f}{-np.tan(azimuth):10.3f}{second:10.3f}{-np.tan(2 * azimuth):11.3f}{transverse_ratio:11.3f}")
    assert np.isclose(first, -np.tan(azimuth), rtol=0.02) and np.isclose(second, -np.tan(2 * azimuth), rtol=0.02)
    assert transverse_ratio < 0.1

 azimuth   Z(m_rp)/Z(m_rt)  -tan(az)   ratio 2  -tan(2az)  T/R, m_rr
      20            -0.364    -0.364    -0.839     -0.839      0.018
     110             2.747     2.747    -0.839     -0.839      0.043
     200            -0.364    -0.364    -0.839     -0.839      0.015
     290             2.747     2.747    -0.839     -0.839      0.046


## Standard sources on the lune

The lune longitude $\gamma$ and latitude $\delta$ of Tape & Tape (2012) place a moment tensor by its eigenvalues alone. A double couple sits at the centre and the isotropic sources at the poles. The CLVD with eigenvalues $(2, -1, -1)$ is at $\gamma = -30°$ and its opposite at $+30°$. A tensile crack with eigenvalues $(3, 1, 1)$ (Poisson's ratio 1/4) is at $(-30°, 60.5°)$.

In [5]:
standard_sources = {"double couple": [0, 1, -1, 0, 0, 0], "explosion": [1, 1, 1, 0, 0, 0],
                    "implosion": [-1, -1, -1, 0, 0, 0], "CLVD (2, -1, -1)": [2, -1, -1, 0, 0, 0],
                    "CLVD (-2, 1, 1)": [-2, 1, 1, 0, 0, 0], "crack (3, 1, 1)": [3, 1, 1, 0, 0, 0]}
expected_deg = {"double couple": (0, 0), "explosion": (None, 90), "implosion": (None, -90),
                "CLVD (2, -1, -1)": (-30, 0), "CLVD (-2, 1, 1)": (30, 0), "crack (3, 1, 1)": (-30, 60.5)}
for name, m6 in standard_sources.items():
    gamma_deg, delta_deg = (float(angle[0]) for angle in mts6_to_gamma_delta(np.array(m6, dtype=float).reshape(1, -1)))
    print(f"{name:18s} gamma {np.round(gamma_deg, 1) + 0.0:6.1f}  delta {np.round(delta_deg, 1) + 0.0:6.1f}")
    expected_gamma, expected_delta = expected_deg[name]
    assert abs(delta_deg - expected_delta) < 0.1
    assert expected_gamma is None or abs(gamma_deg - expected_gamma) < 0.1

double couple      gamma    0.0  delta    0.0
explosion          gamma    0.0  delta   90.0
implosion          gamma    0.0  delta  -90.0
CLVD (2, -1, -1)   gamma  -30.0  delta    0.0
CLVD (-2, 1, 1)    gamma   30.0  delta    0.0
crack (3, 1, 1)    gamma  -30.0  delta   60.5


## The scalar moment

The library's scalar moment is that of Silver & Jordan (1982), $M_0 = \sqrt{\tfrac12 \sum_{ij} M_{ij}^2}$ over the full symmetric tensor, in which each off-diagonal component appears twice. It does not depend on how the tensor is oriented: a strike-slip fault written as $m_{tt} - m_{pp}$ and the same fault rotated by 45° onto $m_{tp}$ have the same moment. The moment magnitude is $M_w = \tfrac23 (\log_{10} M_0 - 9.1)$ with $M_0$ in N m. The prior on magnitude, the moment-tensor scaler, the source time function and the reported magnitudes all use this moment; pyrocko's `MomentTensor.scalar_moment` is the same quantity.

In [ ]:
examples = {"strike-slip as m_tt - m_pp": [0, 1, -1, 0, 0, 0], "strike-slip as m_tp": [0, 0, 0, 0, 0, 1],
            "dip-slip as m_rt": [0, 0, 0, 1, 0, 0], "explosion": [1, 1, 1, 0, 0, 0]}
print(f"{'unit tensor':28s}{'scalar_moment':>15s}{'pyrocko':>10s}")
for name, m6 in examples.items():
    library, reference = scalar_moment(m6), pyrocko_mt(m6).scalar_moment()
    print(f"{name:28s}{library:15.3f}{reference:10.3f}")
    assert np.isclose(library, reference)

## Horizontal components: storage keys and orientation

Event and simulation files store the east component under the key `1` and the north component under `2`. Stations whose horizontals are coded 1 and 2 are not necessarily oriented north and east. Before export, `rotate_horizontals_to_north_east` turns them to north and east using the azimuths in their StationXML. Here a station records a known north and east signal on channels at azimuths 30° and 120°, and the rotation recovers it.

In [7]:
print("storage keys for components ZEN:", component_alias("ZEN"))
signal_rng = np.random.default_rng(0)
vertical_m, north_m, east_m = signal_rng.normal(size=(3, 200))
azimuth_1 = np.radians(30.0)
channel_1 = north_m * np.cos(azimuth_1) + east_m * np.sin(azimuth_1)
channel_2 = -north_m * np.sin(azimuth_1) + east_m * np.cos(azimuth_1)
header = {"network": "XX", "station": "STA", "location": "", "starttime": UTCDateTime(2020, 1, 1), "sampling_rate": 1.0}
recorded = Stream([Trace(data, header={**header, "channel": code})
                   for code, data in [("BHZ", vertical_m), ("BH1", channel_1), ("BH2", channel_2)]])
channels = [Channel(code=code, location_code="", latitude=0, longitude=0, elevation=0, depth=0, azimuth=azimuth_deg,
                    dip=dip_deg, sample_rate=1.0) for code, azimuth_deg, dip_deg in [("BHZ", 0, -90), ("BH1", 30, 0), ("BH2", 120, 0)]]
inventory = Inventory(networks=[Network("XX", stations=[Station("STA", 0, 0, 0, channels=channels)])], source="example")
rotated = {trace.stats.channel: trace.data for trace in rotate_horizontals_to_north_east(recorded, inventory)}
print("channels after rotation:", sorted(rotated))
print("north and east recovered to within 1e-12:", bool(np.abs(rotated["BHN"] - north_m).max() < 1e-12
                                                        and np.abs(rotated["BHE"] - east_m).max() < 1e-12))
assert np.allclose(rotated["BHN"], north_m) and np.allclose(rotated["BHE"], east_m)

storage keys for components ZEN: Z12
channels after rotation: ['BHE', 'BHN', 'BHZ']
north and east recovered to within 1e-12: True
